# CP2 — Time-Aware BSARec (Beauty + ML-1M)

Notebook ini melatih BSARec dengan **time-interval embedding** untuk Beauty dan ML-1M. Jalankan sel **berurutan**. File `.txt` bukan program: unggah pasangan item + bucket, lalu notebook menyambungkannya ke model.

**Input yang perlu diunggah pada sel berikutnya (4 file):**
- `Beauty.txt` + `Beauty_cp2_time_buckets.txt`
- `ML-1M.txt` + `ML-1M_cp2_time_buckets.txt`

Notebook memakai BSARec upstream pada commit `c80bdc0f1a889b78fe9a27e2058389cc84caea84`. Bucket target tidak diberikan ke model; hanya bucket pada history yang dipakai. CP1 files tidak diubah. Training Beauty dan ML-1M dijalankan di sel yang berbeda.

**Catatan:** perlu runtime Google Colab dengan GPU. Sel training memakai konfigurasi CP1 yang tercatat; training penuh dapat memakan waktu. Jangan klik *Run all* sebelum data diunggah.


In [1]:
# Setup: clone source yang dipin ke commit audit, lalu pasang dependensi non-Torch.
%cd /content
!git clone https://github.com/yehjin-shin/BSARec.git
%cd /content/BSARec
!git checkout c80bdc0f1a889b78fe9a27e2058389cc84caea84
%cd /content/BSARec/src
!pip -q install scipy tqdm
import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('PERINGATAN: training akan berjalan di CPU dan kemungkinan sangat lambat.')


/content
Cloning into 'BSARec'...
remote: Enumerating objects: 592, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 592 (delta 28), reused 20 (delta 20), pack-reused 549 (from 1)
Receiving objects: 100% (592/592), 859.10 MiB | 32.46 MiB/s, done.
Resolving deltas: 100% (136/136), done.
Updating files: 100% (384/384), done.
/content/BSARec
Note: switching to 'c80bdc0f1a889b78fe9a27e2058389cc84caea84'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now a

In [5]:
from pathlib import Path
import shutil

root = Path("/content/BSARec/src")
data_dir = root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

pairs = {
    "Beauty.txt": "Beauty.txt",
    "Beauty_cp2_time_buckets (1).txt": "Beauty_cp2_time_buckets.txt",
    "ML-1M.txt": "ML-1M.txt",
    "ML-1M_cp2_time_buckets (1).txt": "ML-1M_cp2_time_buckets.txt",
}

for uploaded_name, final_name in pairs.items():
    source = root / uploaded_name
    if not source.exists():
        raise FileNotFoundError(f"Tidak ditemukan: {source}")
    shutil.copy2(source, data_dir / final_name)
    print(f"Siap: {final_name}")


Siap: Beauty.txt
Siap: Beauty_cp2_time_buckets.txt
Siap: ML-1M.txt
Siap: ML-1M_cp2_time_buckets.txt


In [6]:
%%writefile /content/BSARec/src/cp2_time_runtime.py
import os
import torch
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler
from dataset import RecDataset


def read_time_buckets(args, seq_dic):
    item_path = args.data_dir + args.data_name + '.txt'
    time_path = os.path.join(args.data_dir, args.data_name + '_cp2_time_buckets.txt')
    item_rows = []
    with open(item_path, encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            fields = line.split()
            if len(fields) < 2:
                raise ValueError(f'{item_path}:{line_no}: row kosong/pendek')
            item_rows.append((int(fields[0]), [int(v) for v in fields[1:]]))
    time_rows = []
    with open(time_path, encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            fields = line.split()
            if len(fields) < 2:
                raise ValueError(f'{time_path}:{line_no}: row kosong/pendek')
            time_rows.append((int(fields[0]), [int(v) for v in fields[1:]]))
    if len(item_rows) != len(time_rows) or len(item_rows) != len(seq_dic['user_seq']):
        raise ValueError('Jumlah baris item/time/model berbeda')
    time_sequences = []
    for i, ((item_uid, items), (time_uid, buckets), model_items) in enumerate(
        zip(item_rows, time_rows, seq_dic['user_seq']), 1
    ):
        if item_uid != time_uid:
            raise ValueError(f'User ID tidak cocok pada baris {i}: item={item_uid}, time={time_uid}')
        if len(items) != len(buckets) or len(items) != len(model_items):
            raise ValueError(f'Panjang item/bucket tidak cocok pada baris {i}')
        if any(bucket < 0 or bucket > 7 for bucket in buckets):
            raise ValueError(f'ID bucket di luar 0..7 pada baris {i}')
        time_sequences.append(buckets)
    return time_sequences


class TimeRecDataset(RecDataset):
    '''Keeps the upstream split/windows and mirrors them for time IDs.'''
    def __init__(self, args, user_seq, time_seq, data_type='train'):
        if len(user_seq) != len(time_seq):
            raise ValueError('Jumlah sequence item/time berbeda')
        self.time_user_seq = []
        self.max_len = args.max_seq_length
        super().__init__(args, user_seq, data_type=data_type)
        if data_type == 'train':
            for items, times in zip(user_seq, time_seq):
                item_prefix = items[-(self.max_len + 2):-2]
                time_prefix = times[-(self.max_len + 2):-2]
                if len(item_prefix) != len(time_prefix):
                    raise ValueError('Panjang training prefix item/time berbeda')
                for end in range(1, len(item_prefix) + 1):
                    self.time_user_seq.append(time_prefix[:end])
        elif data_type == 'valid':
            self.time_user_seq = [times[:-1] for times in time_seq]
        else:
            self.time_user_seq = [list(times) for times in time_seq]
        if len(self.time_user_seq) != len(self.user_seq):
            raise ValueError(f'Jumlah window item/time berbeda: {len(self.user_seq)} vs {len(self.time_user_seq)}')

    def __getitem__(self, index):
        base = super().__getitem__(index)
        history_times = self.time_user_seq[index][:-1]  # target-position time is excluded
        if len(history_times) > self.max_len:
            history_times = history_times[-self.max_len:]
        pad_len = self.max_len - len(history_times)
        padded_times = [0] * pad_len + history_times
        if len(padded_times) != self.max_len:
            raise AssertionError('Panjang time tensor salah')
        return (*base, torch.tensor(padded_times, dtype=torch.long))


def get_time_dataloder(args, seq_dic):
    time_seq = read_time_buckets(args, seq_dic)
    train_ds = TimeRecDataset(args, seq_dic['user_seq'], time_seq, data_type='train')
    eval_ds = TimeRecDataset(args, seq_dic['user_seq'], time_seq, data_type='valid')
    test_ds = TimeRecDataset(args, seq_dic['user_seq'], time_seq, data_type='test')
    train_loader = DataLoader(train_ds, sampler=RandomSampler(train_ds), batch_size=args.batch_size, num_workers=args.num_workers)
    eval_loader = DataLoader(eval_ds, sampler=SequentialSampler(eval_ds), batch_size=args.batch_size, num_workers=args.num_workers)
    test_loader = DataLoader(test_ds, sampler=SequentialSampler(test_ds), batch_size=args.batch_size, num_workers=args.num_workers)
    return train_loader, eval_loader, test_loader


Writing /content/BSARec/src/cp2_time_runtime.py


In [7]:
# Tambahkan bucket waktu ke dataset, trainer, dan embedding BSARec.
from pathlib import Path
import re

src = Path('/content/BSARec/src')

def replace_once(path, old, new):
    text = path.read_text(encoding='utf-8')
    count = text.count(old)
    if count != 1:
        raise RuntimeError(f'{path.name}: expected exactly one anchor, found {count}: {old!r}')
    path.write_text(text.replace(old, new, 1), encoding='utf-8')

# Preserve upstream split/evaluator, substitute only the loader implementation.
dataset_path = src / 'dataset.py'
dataset_text = dataset_path.read_text(encoding='utf-8')
match = re.search(r'(?ms)^def get_dataloder\(args,\s*seq_dic\):.*\Z', dataset_text)
if not match:
    raise RuntimeError('Tidak menemukan fungsi get_dataloder upstream; patch dihentikan tanpa mengubah kode.')
dataset_text = dataset_text[:match.start()] + '''def get_dataloder(args, seq_dic):
    from cp2_time_runtime import get_time_dataloder
    return get_time_dataloder(args, seq_dic)
'''
dataset_path.write_text(dataset_text, encoding='utf-8')

# Pass the parallel time tensor through the existing BSARec training/evaluation loop.
trainer_path = src / 'trainers.py'
replace_once(trainer_path,
    'user_ids, input_ids, answers, neg_answer, same_target = batch',
    'user_ids, input_ids, answers, neg_answer, same_target, time_ids = batch')
replace_once(trainer_path,
    'self.model.calculate_loss(input_ids, answers, neg_answer, same_target, user_ids)',
    'self.model.calculate_loss(input_ids, answers, neg_answer, same_target, user_ids, time_ids=time_ids)')
replace_once(trainer_path,
    'user_ids, input_ids, answers, _, _ = batch',
    'user_ids, input_ids, answers, _, _, time_ids = batch')
replace_once(trainer_path,
    'self.model.predict(input_ids, user_ids)',
    'self.model.predict(input_ids, user_ids, time_ids=time_ids)')

# Add a trainable 8-row time embedding; padded item positions get zero time contribution.
model_path = src / 'model' / 'bsarec.py'
replace_once(model_path,
    '        self.dropout = nn.Dropout(args.hidden_dropout_prob)\n',
    '        self.dropout = nn.Dropout(args.hidden_dropout_prob)\n        self.time_embeddings = nn.Embedding(8, args.hidden_size)\n')
replace_once(model_path,
    '    def forward(self, input_ids, user_ids=None, all_sequence_output=False):',
    '    def forward(self, input_ids, user_ids=None, all_sequence_output=False, time_ids=None):')
replace_once(model_path,
    '        sequence_emb = self.add_position_embedding(input_ids)',
    '        sequence_emb = self.add_time_position_embedding(input_ids, time_ids)')
helper = '''    def add_time_position_embedding(self, input_ids, time_ids):
        if time_ids is None:
            raise ValueError("CP2 BSARec requires time bucket IDs")
        seq_len = input_ids.size(1)
        position_ids = torch.arange(seq_len, dtype=torch.long, device=input_ids.device)
        position_ids = position_ids.unsqueeze(0).expand_as(input_ids)
        item_emb = self.item_embeddings(input_ids)
        position_emb = self.position_embeddings(position_ids)
        time_emb = self.time_embeddings(time_ids)
        time_emb = time_emb * input_ids.ne(0).unsqueeze(-1).to(time_emb.dtype)
        sequence_emb = item_emb + position_emb + time_emb
        sequence_emb = self.LayerNorm(sequence_emb)
        return self.dropout(sequence_emb)

'''
forward_anchor = '    def forward(self, input_ids, user_ids=None, all_sequence_output=False, time_ids=None):'
text = model_path.read_text(encoding='utf-8')
if text.count(forward_anchor) != 1:
    raise RuntimeError('Anchor forward BSARec tidak unik')
model_path.write_text(text.replace(forward_anchor, helper + forward_anchor, 1), encoding='utf-8')
replace_once(model_path,
    '    def calculate_loss(self, input_ids, answers, neg_answers, same_target, user_ids):',
    '    def predict(self, input_ids, user_ids=None, time_ids=None):\n        return self.forward(input_ids, user_ids, False, time_ids)\n\n    def calculate_loss(self, input_ids, answers, neg_answers, same_target, user_ids, time_ids=None):')
replace_once(model_path,
    '        seq_output = self.forward(input_ids)',
    '        seq_output = self.forward(input_ids, time_ids=time_ids)')
print('Patch CP2 diterapkan.')


Patch CP2 diterapkan.


In [8]:
# Syntax check dan smoke test singkat: loader + forward/loss BSARec.
!python -m py_compile dataset.py trainers.py model/bsarec.py cp2_time_runtime.py
from types import SimpleNamespace
import torch
from dataset import get_seq_dic
from cp2_time_runtime import read_time_buckets, TimeRecDataset
from model.bsarec import BSARecModel

args = SimpleNamespace(
    data_name='Beauty', data_dir='./data/', max_seq_length=50, model_type='BSARec',
    batch_size=2, num_workers=0, hidden_size=64, hidden_dropout_prob=0.1,
    num_hidden_layers=2, num_attention_heads=1, attention_probs_dropout_prob=0.1,
    hidden_act='gelu', initializer_range=0.02, alpha=0.7, c=5,
)
seq_dic, max_item, num_users = get_seq_dic(args)
args.item_size = max_item + 1
args.num_users = num_users + 1
time_seq = read_time_buckets(args, seq_dic)
small_ds = TimeRecDataset(args, seq_dic['user_seq'][:3], time_seq[:3], data_type='test')
examples = [small_ds[i] for i in range(min(2, len(small_ds)))]
batch = [torch.stack([ex[j] for ex in examples]) for j in range(6)]
model = BSARecModel(args)
loss = model.calculate_loss(batch[1], batch[2], batch[3], batch[4], batch[0], time_ids=batch[5])
assert torch.isfinite(loss).item(), loss
assert batch[5].shape == batch[1].shape == (len(examples), 50)
print('Smoke test OK | loss finite | input/time shape:', tuple(batch[1].shape), tuple(batch[5].shape))


Smoke test OK | loss finite | input/time shape: (2, 50) (2, 50)


## Training

Jalankan sel yang kamu butuhkan. Kalau ingin langsung mengerjakan keduanya, jalankan sel Beauty lalu sel ML-1M. Masing-masing menyimpan checkpoint `.pt` dan log `.log` di `src/output/`. Konfigurasi memakai seed 42 dan nilai CP1 yang tercatat.

Catatan: ini menjalankan **BSARec + time embedding**, bukan SASRec. Jangan jalankan dua sel training secara bersamaan.


In [9]:
# Run Beauty — baseline CP1 config: lr=0.0005, alpha=0.7, c=5, 1 head, seed 42.
!python main.py --data_name Beauty --model_type BSARec --lr 0.0005 --alpha 0.7 --c 5 --num_attention_heads 1 --seed 42 --train_name BSARec_T_Beauty_s42 --num_workers 4


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-11-20 08:24:44,608 - Namespace(data_dir='./data/', output_dir='output/', data_name='Beauty', do_eval=False, load_model=None, train_name='BSARec_T_Beauty_s42', num_items=10, num_users=22364, lr=0.0005, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=4, seed=42, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='BSARec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=1, attention_probs_dropout_prob

In [10]:
# Run ML-1M — CP1 config: lr=0.001, batch 256, hidden 64, 2 layers, 2 heads, seed 42.
!python main.py --data_name ML-1M --model_type BSARec --lr 0.001 --alpha 0.9 --c 3 --batch_size 256 --hidden_size 64 --num_hidden_layers 2 --num_attention_heads 2 --seed 42 --train_name BSARec_T_ML1M_s42 --num_workers 2


2026-11-20 08:41:23,598 - Namespace(data_dir='./data/', output_dir='output/', data_name='ML-1M', do_eval=False, load_model=None, train_name='BSARec_T_ML1M_s42', num_items=10, num_users=6041, lr=0.001, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=2, seed=42, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='BSARec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=2, attention_probs_dropout_prob=0.5, hidden_dropout_prob=0.5, initializer_range=0.02, c=3, alpha=0.9, cuda_condition=True, data_file='./data/ML-1M.txt', item_size=3417, checkpoint_path='output/BSARec_T_ML1M_s42.pt', same_target_path='./data/ML-1M_same_target.npy')
2026-11-20 08:41:23,618 - BSARecModel(
  (item_embeddings): Embedding(3417, 64, padding_idx=0)
  (position_embeddings): Embedding(50, 64)
  (LayerNorm): LayerNorm()
  (dropout): Dropout(p=0.5, inplace=False)
  (time_embeddings): Embedding(8, 64)
  (i

In [1]:
# Optional: copy outputs to Google Drive so Colab runtime reset won't remove them.
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
save_dir = Path('/content/drive/MyDrive/CP2_BSARec_TimeAware')
save_dir.mkdir(parents=True, exist_ok=True)
for name in ['BSARec_T_Beauty_s42', 'BSARec_T_ML1M_s42']:
    for suffix in ['.log', '.pt']:
        src_file = Path('output') / (name + suffix)
        if src_file.exists():
            shutil.copy2(src_file, save_dir / src_file.name)
print('Output yang sudah ada disalin ke', save_dir)


Mounted at /content/drive
Output yang sudah ada disalin ke /content/drive/MyDrive/CP2_BSARec_TimeAware
